## 1. Setup & Config

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaModel, RobertaTokenizerFast
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import GroupKFold, KFold
from scipy.stats import pearsonr
import numpy as np
import pandas as pd
import json, os, math, time, copy
from typing import Sequence, Optional, Dict, Tuple, Any, List
from torch.optim.lr_scheduler import LambdaLR

In [ ]:
!pip install "torchao>=0.16.0" -q

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
CFG = {
    'model_name':     'FacebookAI/roberta-large',
    'max_len':        128,
    'batch_size':     4,
    'epochs':         10,
    'lr':             2e-5,
    'weight_decay':   0.01,
    'lora_r':         8,
    'lora_alpha':     32,
    'lora_dropout':   0.1,
    'time_dim':       64,
    'user_emb_dim':   32,
    'hidden_dim':     256,
    'm_history':      5,
    'n_forecast':     1,

    'tf_p_start':     1.0,
    'tf_p_end':       0.1,

    'train_csv':      "https://drive.google.com/uc?id=1GjgeZ0BdBhWm8t463WJ6a5DJRo4FUV8Z",
    'test_csv':       "https://drive.google.com/uc?id=1RtKN8y7MUsEDk4j2UbLOe3q4mAGBQi3w",
    'marker_csv':     "https://drive.google.com/uc?id=1FQYogNWXbqnR1VHtYOnVRXQfykSSUPMc",
    'output_csv':     'task2a_predictions_fixed_paper.csv',
    'result_csv':     "https://drive.google.com/uc?id=1R7edcVHvyTSrRvYkeB1jQvVX3Gf5riwd",
}

## 2. Data Preparation

In [ ]:
def norm_valence(v):   return v / 2.0
def norm_arousal(a):   return a - 1.0
def denorm_valence(v): return v * 2.0
def denorm_arousal(a): return a + 1.0

def norm_delta_v(dv):   return dv / 2.0
def norm_delta_a(da):   return da / 2.0
def denorm_delta_v(dv): return dv * 2.0
def denorm_delta_a(da): return da

In [ ]:
train_raw  = pd.read_csv(CFG['train_csv'])
test_raw   = pd.read_csv(CFG['test_csv'])
marker_raw = pd.read_csv(CFG['marker_csv'])

train_raw['timestamp']  = pd.to_datetime(train_raw['timestamp'])
marker_raw['timestamp'] = pd.to_datetime(marker_raw['timestamp'])
test_raw['timestamp_min'] = pd.to_datetime(test_raw['timestamp_min'])
test_raw['timestamp_max'] = pd.to_datetime(test_raw['timestamp_max'])

train_raw  = train_raw.sort_values(['user_id','timestamp']).reset_index(drop=True)
marker_raw = marker_raw.sort_values(['user_id','timestamp']).reset_index(drop=True)

for df in [train_raw, marker_raw]:
    df['valence'] = df['valence'].apply(norm_valence)
    df['arousal'] = df['arousal'].apply(norm_arousal)
    df['state_change_valence'] = pd.to_numeric(df['state_change_valence'], errors='coerce').apply(
        lambda x: norm_delta_v(x) if pd.notna(x) else np.nan
    )
    df['state_change_arousal'] = pd.to_numeric(df['state_change_arousal'], errors='coerce').apply(
        lambda x: norm_delta_a(x) if pd.notna(x) else np.nan
    )

In [ ]:
print(f'Train rows: {len(train_raw)} | users: {train_raw.user_id.nunique()}')
print(f'Marker rows: {len(marker_raw)} | forecasting users: {marker_raw[marker_raw.is_forecasting_user==True].user_id.nunique()}')
print(f'Test rows: {len(test_raw)}')

In [ ]:
result_raw = pd.read_csv(CFG['result_csv'])
print('gold result_raw:', result_raw.shape, '| kolom:', list(result_raw.columns))

In [ ]:
all_known_users = sorted(set(
    train_raw['user_id'].unique().tolist() +
    marker_raw['user_id'].unique().tolist()
))

user2idx = {uid: i + 1 for i, uid in enumerate(all_known_users)}
COLD_START_IDX = 0
NUM_USERS = len(user2idx) + 1

def get_user_idx(uid):
    """user lookup, unseen users map to a dedicated cold-start embedding."""
    return user2idx.get(uid, COLD_START_IDX)

train_raw['user_idx']  = train_raw['user_id'].apply(get_user_idx)
marker_raw['user_idx'] = marker_raw['user_id'].apply(get_user_idx)

print(f'Total embedding slots: {NUM_USERS}  (slot 0 = cold-start)')
print(f'Known users: {len(user2idx)}')

test_coverage = [uid in user2idx for uid in test_raw['user_id']]
print(f'Test users in user2idx: {sum(test_coverage)}/{len(test_coverage)}')

In [ ]:
def compute_delta_time_days(timestamps: List) -> np.ndarray:
    times = pd.to_datetime(timestamps)
    delta = pd.Series(times).diff().dt.total_seconds().fillna(0) / 86400.0
    return delta.values.astype(np.float32)

In [ ]:
def build_train_sequences(df: pd.DataFrame, m: int, n: int = 1) -> List[Dict]:
    samples = []
    for user_id, g in df.groupby('user_id'):
        g = g.sort_values('timestamp').reset_index(drop=True)
        for t in range(m - 1, len(g) - n):
            hist = g.iloc[t - m + 1 : t + 1]
            fut  = g.iloc[t + 1 : t + 1 + n]

            if fut[['state_change_valence','state_change_arousal']].isna().any().any():
                continue

            samples.append({
                'user_id':       user_id,
                'user_idx':      int(hist.iloc[0]['user_idx']),
                'texts':         hist['text'].fillna('').tolist(),
                'timestamps':    hist['timestamp'].tolist(),
                'va_history':    hist[['valence','arousal']].values.astype(np.float32),
                'delta_targets': fut[['state_change_valence','state_change_arousal']].values.astype(np.float32),
                'last_va':       hist[['valence','arousal']].iloc[-1].values.astype(np.float32),
            })
    return samples

In [ ]:
train_samples = build_train_sequences(train_raw, m=CFG['m_history'], n=CFG['n_forecast'])
print('train_samples:', len(train_samples))

In [ ]:
def build_test_sequences(marker_df: pd.DataFrame, test_df: pd.DataFrame, m: int) -> List[Dict]:
    samples = []
    for _, row in test_df.iterrows():
        uid      = row['user_id']
        user_idx = get_user_idx(uid)

        g = marker_df[
            (marker_df['user_id'] == uid) &
            (marker_df['timestamp'] >= row['timestamp_min']) &
            (marker_df['timestamp'] <= row['timestamp_max'])
        ].sort_values('timestamp')

        if len(g) == 0:
            texts  = [''] * m
            va     = np.zeros((m, 2), dtype=np.float32)
            times  = [row['timestamp_min']] * m
        elif len(g) < m:
            last  = g.iloc[-1]
            pad_n = m - len(g)
            texts = g['text'].fillna('').tolist() + [last['text']] * pad_n
            va    = np.vstack([
                g[['valence','arousal']].values,
                np.tile(last[['valence','arousal']].values, (pad_n, 1))
            ]).astype(np.float32)
            times = g['timestamp'].tolist() + [last['timestamp']] * pad_n
        else:
            g     = g.iloc[-m:]
            texts = g['text'].fillna('').tolist()
            va    = g[['valence','arousal']].values.astype(np.float32)
            times = g['timestamp'].tolist()

        samples.append({
            'user_id':    uid,
            'user_idx':   user_idx,
            'texts':      texts,
            'timestamps': times,
            'va_history': va,
            'last_va':    va[-1],
        })
    return samples

In [ ]:
test_samples = build_test_sequences(marker_raw, test_raw, m=CFG['m_history'])
print('test_samples:', len(test_samples))

## 3. Model Architecture

In [ ]:
class VAForecastDataset(Dataset):
    def __init__(self, samples: List[Dict], tokenizer, max_len: int = 128):
        self.samples   = samples
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        enc = self.tokenizer(
            s['texts'],
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        delta_time = compute_delta_time_days(s['timestamps'])

        item = {
            'input_ids':      enc['input_ids'],
            'attention_mask': enc['attention_mask'],
            'delta_time':     torch.tensor(delta_time, dtype=torch.float),
            'va_history':     torch.tensor(s['va_history'], dtype=torch.float),
            'user_idx':       torch.tensor(s['user_idx'], dtype=torch.long),
            'last_va':        torch.tensor(s['last_va'], dtype=torch.float),
        }
        if 'delta_targets' in s:
            item['delta_target'] = torch.tensor(s['delta_targets'], dtype=torch.float)
        return item

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim: int = 64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(1, dim), nn.ReLU(), nn.Linear(dim, dim)
        )
    def forward(self, delta_t):
        return self.mlp(delta_t.unsqueeze(-1))

In [ ]:
class FixedVAForecaster(nn.Module):
    def __init__(self, num_users: int, m: int = 5, n: int = 1,
                 user_emb_dim: int = 32, time_emb_dim: int = 64,
                 hidden_dim: int = 256):
        super().__init__()
        self.m = m
        self.n = n

        roberta = RobertaModel.from_pretrained('FacebookAI/roberta-large')
        lora_cfg = LoraConfig(
            r=8, lora_alpha=32,
            target_modules=['query', 'value'],
            lora_dropout=0.1, bias='none'
        )
        self.roberta   = get_peft_model(roberta, lora_cfg)
        text_dim       = self.roberta.config.hidden_size   # 1024

        self.time_emb = TimeEmbedding(time_emb_dim)
        self.user_emb = nn.Embedding(num_users, user_emb_dim, padding_idx=0)

        fused_dim    = text_dim + time_emb_dim + 2 + user_emb_dim
        self.fusion  = nn.Linear(fused_dim, hidden_dim)

        self.temporal = nn.GRU(hidden_dim, hidden_dim, batch_first=True)

        self.decoder   = nn.GRU(hidden_dim + 2, hidden_dim, batch_first=True)
        self.delta_head = nn.Linear(hidden_dim, 2)

    def encode_text(self, input_ids, attention_mask):
        B, M, L = input_ids.shape
        out = self.roberta(
            input_ids=input_ids.view(B * M, L),
            attention_mask=attention_mask.view(B * M, L)
        )
        mask      = attention_mask.view(B * M, L).unsqueeze(-1).float()
        mean_pool = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        return mean_pool.view(B, M, -1)

    def forward(self, input_ids, attention_mask, delta_time,
                va_history, user_idx, tf_prob: float = 0.0,
                va_target: Optional[torch.Tensor] = None):
        B = input_ids.size(0)
        text_emb = self.encode_text(input_ids, attention_mask)
        t_emb = self.time_emb(delta_time)
        u_emb = self.user_emb(user_idx).unsqueeze(1).expand(-1, self.m, -1)
        fused = torch.relu(self.fusion(
            torch.cat([text_emb, t_emb, va_history, u_emb], dim=-1)
        ))
        _, h = self.temporal(fused)

        preds   = []
        last_va = va_history[:, -1, :]
        for step in range(self.n):
            dec_in = torch.cat([h.transpose(0, 1), last_va.unsqueeze(1)], dim=-1)
            out, h = self.decoder(dec_in, h)
            delta  = torch.tanh(self.delta_head(out.squeeze(1)))
            next_va = last_va + delta
            preds.append(delta)
            if tf_prob > 0.0 and va_target is not None and step < va_target.size(1):
                use_teacher = (torch.rand(B, device=input_ids.device) < tf_prob)
                gt_va       = va_target[:, step, :]
                last_va     = torch.where(
                    use_teacher.unsqueeze(1).expand_as(next_va), gt_va, next_va
                )
            else:
                last_va = next_va
        return torch.stack(preds, dim=1)

## 4. Training & Evaluation Setup

In [ ]:
def _pearson(x: Sequence[float], y: Sequence[float]) -> Tuple[float, Optional[float]]:
    x_arr = np.asarray(x, dtype=float)
    y_arr = np.asarray(y, dtype=float)
    if x_arr.size < 2 or np.all(x_arr == x_arr[0]) or np.all(y_arr == y_arr[0]):
        return float('nan'), None
    r, p = pearsonr(x_arr, y_arr)
    return float(r), float(p)


def task2_correlation(
    user_ids: Optional[Sequence[Any]],
    predictions: Sequence[float],
    labels: Sequence[float],
) -> Dict[str, Any]:
    r, p = _pearson(predictions, labels)
    mae = float(np.mean(np.abs(np.asarray(predictions) - np.asarray(labels))))
    return {'r': r, 'p': p, 'mae': mae}

In [ ]:
def ccc_loss(pred: torch.Tensor, target: torch.Tensor,
             eps: float = 1e-8) -> torch.Tensor:
    pred_mean   = pred.mean()
    target_mean = target.mean()
    pred_var    = pred.var(unbiased=False)
    target_var  = target.var(unbiased=False)
    covariance  = ((pred - pred_mean) * (target - target_mean)).mean()
    ccc = (2.0 * covariance) / (
        pred_var + target_var + (pred_mean - target_mean) ** 2 + eps
    )
    return 1.0 - ccc


def combined_ccc_mae_loss(pred_v, pred_a, true_v, true_a,
                          w_ccc: float = 0.7,
                          w_mae: float = 0.3) -> torch.Tensor:
    loss_ccc = ccc_loss(pred_v, true_v) + ccc_loss(pred_a, true_a)
    loss_mae = (torch.abs(pred_v - true_v).mean() +
                torch.abs(pred_a - true_a).mean())
    return w_ccc * loss_ccc + w_mae * loss_mae

In [ ]:
def freeze_backbone(model):
    """Freeze all RoBERTa parameters (including LoRA adapters)"""
    for name, param in model.named_parameters():
        if 'roberta' in name:
            param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  [freeze] trainable params: {trainable:,}')


def unfreeze_backbone(model):
    """Unfreeze only the LoRA adapter weights inside RoBERTa"""
    for name, param in model.named_parameters():
        if 'lora_' in name:
            param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  [unfreeze LoRA] trainable params: {trainable:,}')


def build_stage1_optimizer(model):
    params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(params, lr=1e-3, weight_decay=0.01)


def build_stage2_optimizer(model):
    backbone_params, head_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'roberta' in name:
            backbone_params.append(param)
        else:
            head_params.append(param)
    return torch.optim.AdamW([
        {'params': backbone_params, 'lr': 5e-6},
        {'params': head_params,     'lr': 1e-4},
    ], weight_decay=0.01)


def build_warmup_scheduler(optimizer, n_warmup: int, n_total: int):
    def lr_lambda(step):
        if step < n_warmup:
            return float(step) / max(n_warmup, 1)
        progress = (step - n_warmup) / max(n_total - n_warmup, 1)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return LambdaLR(optimizer, lr_lambda)


def get_tf_prob(epoch, total_epochs, p_start=1.0, p_end=0.1):
    progress = epoch / max(total_epochs - 1, 1)
    return p_start + progress * (p_end - p_start)

### Ablation Setup

In [ ]:
class AblatableForecaster2A(FixedVAForecaster):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        hidden_dim = self.fusion.out_features
        # Extra head, used ONLY when 'no_temporal' is active: bypasses BOTH
        # self.temporal and self.decoder (no recurrence at all), single linear
        # regression straight from the fused window representation + last VA.
        self.notemporal_head = nn.Linear(hidden_dim + 2, 2)

    def forward(self, input_ids, attention_mask, delta_time,
                va_history, user_idx, tf_prob: float = 0.0,
                va_target=None, abl=frozenset()):
        """
        Only two ablation flags are recognized here (per the 3 requested configs):
          'no_user'     -> zero out the user embedding before fusion
          'no_temporal' -> skip BOTH GRUs entirely, single-shot linear regression
        'only_text' = {'no_user', 'no_temporal'} combined.
        """
        B = input_ids.size(0)

        t_emb = self.time_emb(delta_time)

        u_emb = self.user_emb(user_idx).unsqueeze(1).expand(-1, self.m, -1)
        if 'no_user' in abl:
            u_emb = torch.zeros_like(u_emb)

        text_emb = self.encode_text(input_ids, attention_mask)

        fused = torch.relu(self.fusion(
            torch.cat([text_emb, t_emb, va_history, u_emb], dim=-1)
        ))  # [B, m, H]

        last_va = va_history[:, -1, :]  # anchor is never ablated

        if 'no_temporal' in abl:
            # literal "no temporal": no GRU at all, single-shot regression
            last_fused = fused[:, -1, :]
            reg_in     = torch.cat([last_fused, last_va], dim=-1)
            delta      = torch.tanh(self.notemporal_head(reg_in))
            return delta.unsqueeze(1)   # [B, 1, 2]

        # normal path: temporal encoder GRU + autoregressive decoder GRU
        _, h = self.temporal(fused)
        preds = []
        for step in range(self.n):
            dec_in = torch.cat([h.transpose(0, 1), last_va.unsqueeze(1)], dim=-1)
            out, h = self.decoder(dec_in, h)
            delta  = torch.tanh(self.delta_head(out.squeeze(1)))
            next_va = last_va + delta
            preds.append(delta)
            if tf_prob > 0.0 and va_target is not None and step < va_target.size(1):
                use_teacher = (torch.rand(B, device=input_ids.device) < tf_prob)
                last_va = torch.where(use_teacher.unsqueeze(1).expand_as(next_va),
                                      va_target[:, step, :], next_va)
            else:
                last_va = next_va
        return torch.stack(preds, dim=1)

In [ ]:
def train_epoch_abl(model, loader, optimizer, scheduler, epoch, total_epochs, abl=frozenset()):
    model.train(); tf_prob = get_tf_prob(epoch, total_epochs); total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        va_target = batch['last_va'].unsqueeze(1).to(device)
        preds = model(input_ids=batch['input_ids'].to(device),
                      attention_mask=batch['attention_mask'].to(device),
                      delta_time=batch['delta_time'].to(device),
                      va_history=batch['va_history'].to(device),
                      user_idx=batch['user_idx'].to(device),
                      tf_prob=tf_prob, va_target=va_target, abl=abl)
        target = batch['delta_target'].to(device)
        loss = combined_ccc_mae_loss(preds[:,0,0], preds[:,0,1], target[:,0,0], target[:,0,1])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step(); scheduler.step(); total_loss += loss.item()
    return total_loss / max(len(loader), 1), tf_prob


@torch.no_grad()
def validate_abl(model, loader, abl=frozenset()):
    model.eval(); pv,pa,tv,ta,uids = [],[],[],[],[]
    for batch in loader:
        preds = model(input_ids=batch['input_ids'].to(device),
                      attention_mask=batch['attention_mask'].to(device),
                      delta_time=batch['delta_time'].to(device),
                      va_history=batch['va_history'].to(device),
                      user_idx=batch['user_idx'].to(device), tf_prob=0.0, abl=abl)
        p = preds[:,0,:].cpu().numpy(); y = batch['delta_target'][:,0,:].numpy()
        pv += p[:,0].tolist(); tv += y[:,0].tolist(); pa += p[:,1].tolist(); ta += y[:,1].tolist()
        uids += batch['user_idx'].cpu().tolist()
    return {'valence': task2_correlation(uids, pv, tv), 'arousal': task2_correlation(uids, pa, ta)}


In [ ]:
@torch.no_grad()
def predict_test_abl(model, test_samples: List[Dict], tokenizer, abl=frozenset()) -> pd.DataFrame:
    model.eval()
    results = []
    test_ds = VAForecastDataset(test_samples, tokenizer, CFG['max_len'])
    loader  = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False)

    sample_idx = 0
    for batch in loader:
        preds = model(
            input_ids      = batch['input_ids'].to(device),
            attention_mask = batch['attention_mask'].to(device),
            delta_time     = batch['delta_time'].to(device),
            va_history     = batch['va_history'].to(device),
            user_idx       = batch['user_idx'].to(device),
            tf_prob        = 0.0,
            abl            = abl,     # <-- the fix
        )
        delta_pred = preds[:, 0, :].cpu().numpy()
        last_va    = batch['last_va'].cpu().numpy()

        for b in range(delta_pred.shape[0]):
            s = test_samples[sample_idx]
            dv_norm = float(delta_pred[b, 0]); da_norm = float(delta_pred[b, 1])
            lv_norm = float(last_va[b, 0]);    la_norm = float(last_va[b, 1])
            delta_v = denorm_delta_v(dv_norm); delta_a = denorm_delta_a(da_norm)
            last_v  = denorm_valence(lv_norm); last_a  = denorm_arousal(la_norm)
            pred_next_v = float(np.clip(last_v + delta_v, -2.0, 2.0))
            pred_next_a = float(np.clip(last_a + delta_a,  0.0, 2.0))
            results.append({
                'user_id': s['user_id'], 'user_idx': s['user_idx'],
                'last_observed_valence': round(last_v, 4),
                'last_observed_arousal': round(last_a, 4),
                'state_change_valence':  round(delta_v, 4),
                'state_change_arousal':  round(delta_a, 4),
                'predicted_next_valence': round(pred_next_v, 4),
                'predicted_next_arousal': round(pred_next_a, 4),
                'cold_start': s['user_idx'] == COLD_START_IDX,
            })
            sample_idx += 1
    return pd.DataFrame(results)


def build_last_observed():
    mt = train_raw.groupby('user_id')['timestamp'].max().reset_index()
    mt = pd.merge(mt, train_raw[['user_id','timestamp','valence','arousal']],
                  on=['user_id','timestamp'], how='left')
    return mt[mt['user_id'].isin(test_raw['user_id'].unique())][['user_id','valence','arousal']]


def postprocess(pred_df):
    df = pred_df.copy(); lo = build_last_observed()
    df = df.drop(columns=[c for c in ['last_observed_valence','last_observed_arousal',
                                      'state_change_valence','state_change_arousal'] if c in df.columns])
    df = pd.merge(df, lo, on='user_id', how='left')
    df['last_observed_valence']=df['valence']; df['last_observed_arousal']=df['arousal']
    df = df.drop(columns=['valence','arousal'])
    df['state_change_valence'] = df['predicted_next_valence'] - df['last_observed_valence']
    df['state_change_arousal'] = df['predicted_next_arousal'] - df['last_observed_arousal']
    return df


def score_vs_gold(df):
    g = result_raw[['user_id','state_change_valence','state_change_arousal']].rename(
        columns={'state_change_valence':'gold_v','state_change_arousal':'gold_a'})
    m = pd.merge(df[['user_id','state_change_valence','state_change_arousal']], g, on='user_id')
    rv = task2_correlation(m['user_id'], m['state_change_valence'], m['gold_v'])
    ra = task2_correlation(m['user_id'], m['state_change_arousal'], m['gold_a'])
    return rv, ra, len(m)


In [ ]:
def compare_table_v2():
    if not os.path.exists(ABL_CSV):
        print('belum ada hasil.'); return
    df = pd.read_csv(ABL_CSV)
    best_per_config = df.loc[df.groupby('config')['avg'].idxmax()]
    print('Best seed per config:')
    print(best_per_config[['config','seed','r_v','r_a','avg','mae_v','mae_a','minutes']]
          .sort_values('avg', ascending=False).to_string(index=False))
    print()
    print('Semua seed (buat cek variance / progress):')
    print(df.sort_values(['config','seed']).to_string(index=False))

## 5. Experiment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/semeval_task2a_results'
for sub in ['ckpt','preds','ablation']:
    os.makedirs(f'{RESULTS_DIR}/{sub}', exist_ok=True)
ABL_CSV = f'{RESULTS_DIR}/ablation/ablation_2a.csv'   # new file, keeps old results untouched

def set_seed(s=42):
    import random; random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print('results:', RESULTS_DIR)

In [ ]:
tokenizer = RobertaTokenizerFast.from_pretrained(CFG['model_name'])
print('tokenizer siap')

In [ ]:
STAGE1_EPOCHS, STAGE2_EPOCHS = 3, 7

_users = np.array(sorted(train_raw['user_id'].unique()))
_tr_u, _va_u = next(iter(KFold(5, shuffle=True, random_state=42).split(_users)))
VAL_USERS = set(_users[_va_u].tolist())
TR = [s for s in train_samples if s['user_id'] not in VAL_USERS]
VA = [s for s in train_samples if s['user_id'] in VAL_USERS]
print(f'train windows={len(TR)} val windows={len(VA)}')

def _score(m):
    rs=[x for x in [m['valence']['r'], m['arousal']['r']] if not np.isnan(x)]
    return float(np.mean(rs)) if rs else -1.0

In [ ]:
def _done():
    if not os.path.exists(ABL_CSV):
        return set()
    d = pd.read_csv(ABL_CSV)
    return set(zip(d['config'], d['seed']))


def run(only=None):
    """
    Same resumable pattern as the original notebook's run_one_config():
    each call trains exactly ONE (config, seed) pair -- whichever is the
    next one not yet in ABL_CSV -- then stops. Call it again to do the next.
    """
    done = _done()
    combo = only
    if combo is None:
        for name, _abl in EXPERIMENTS:
            for seed in SEEDS:
                if (name, seed) not in done:
                    combo = (name, seed)
                    break
            if combo is not None:
                break
    if combo is None:
        print('semua config & seed selesai.')
        return compare_table()

    name, seed = combo
    abl = dict(EXPERIMENTS)[name]
    print(f'>>> {name} seed={seed} \u2014 training...'); t0 = time.time()

    model = train_one(abl, seed)
    pred_df = predict_test_abl(model, test_samples, tokenizer, abl=abl)
    proc = postprocess(pred_df)
    proc.to_csv(f'{RESULTS_DIR}/preds/test_2a_{name}_seed{seed}_proc.csv', index=False)
    torch.save(model.state_dict(), f'{RESULTS_DIR}/ckpt/2a_{name}_seed{seed}.pt')

    rv, ra, n = score_vs_gold(proc)
    avg = float(np.nanmean([rv['r'], ra['r']]))
    minutes = round((time.time() - t0) / 60, 1)
    row = {'config': name, 'seed': seed, 'r_v': round(rv['r'],4), 'r_a': round(ra['r'],4),
           'avg': round(avg,4), 'mae_v': round(rv['mae'],4), 'mae_a': round(ra['mae'],4),
           'n': n, 'minutes': minutes}
    pd.DataFrame([row]).to_csv(ABL_CSV, mode='a', header=not os.path.exists(ABL_CSV), index=False)
    print(f'    r_v={row["r_v"]} r_a={row["r_a"]} avg={row["avg"]} ({minutes}m) -> saved')

    del model
    torch.cuda.empty_cache()
    print(f'=== {name} seed={seed} selesai ===')

In [ ]:
EXPERIMENTS = [
    ('no_user',     {'no_user'}),
    ('no_temporal', {'no_temporal'}),
    ('only_text',   {'no_user', 'no_temporal'}),
]
SEEDS = [42, 152, 2]

In [ ]:
run_one_v2()
compare_table_v2()